**Results check across k**

In [ ]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'
import scienceplots
plt.style.use(['science', 'nature'])

In [ ]:
import numpy as np
import pandas as pd
import arviz as az
# import EM_grouplasso_multimodal
import importlib
import nest_asyncio
nest_asyncio.apply()
# import MCMC_evaluation

In [ ]:
K = 3
N = 500
r = 5
tr = 1

result_dir = f"./Simulation_results/Simulation_1/EM_results/K{K}N{N}r{r}tr{tr}"
# lambda_values = np.logspace(-3, 3, 25).tolist()

ll_list = []
# bic_list = []
for k in range(2,7):

    result = np.load(result_dir + f"/dat_K{k}.npz")
    ll = result["log_likelihood"]
    
    ll_list.append(ll)

df = pd.DataFrame({
    'K': range(2,7),
    'Log-Likelihood': ll_list
})

plt.figure()
plt.plot(df['K'], df['Log-Likelihood'], marker='o', label='Log-Likelihood')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Cross-Validation Log-Likelihood')
# plt.ylim(-97, -75)
# plt.title('Log-Likelihood vs Number of Clusters')
plt.grid(True)
# plt.show()
plt.savefig(f'Fig_EM/Loglik_k{K}n{N}r{r}tr{tr}.pdf', format='pdf', bbox_inches='tight', transparent=True)

In [ ]:
K = 3
N = 200
r = 1
tr = 5

dat_dir = "./Simulation/Simulation_1"
sim_dat = np.load(dat_dir + f"dat_K{K}N{N}r{r}tr{tr}.npz")

result_dir = f"./Simulation_results/Simulation_1/EM_results/K{K}N{N}r{r}tr{tr}/"
result = np.load(result_dir + f"dat_K{K}.npz")

mu_true = sim_dat["mu"]
indice_true = np.argsort(mu_true)
mu_true = mu_true[indice_true]
sigma_true = sim_dat["sigma"][indice_true]

mu_est = np.sort(result["mu"])
indice_est = np.argsort(mu_est)
mu_est = mu_est[indice_est]
sigma_est = result["sigma"][indice_est]

rmse_mu = np.sqrt(np.mean((mu_true - mu_est)**2))
rmse_sigma = np.sqrt(np.mean((sigma_true-sigma_est)**2))
rmse_mu_relative = np.sqrt(np.mean((mu_true - mu_est)**2))/(np.mean(mu_true))
rmse_sigma_relative = np.sqrt(np.mean((sigma_true-sigma_est)**2))/np.mean(sigma_true)
print('RMSE:', rmse_mu, rmse_sigma)
print('RRMSE:', rmse_mu_relative, rmse_sigma_relative)
print(mu_true, sigma_true)
print(mu_est, sigma_est)

truth = sim_dat["w"]
truth = set(np.where(truth==1)[0].tolist())
select = set(result["select"])
select.discard(1000)
print(len(truth))
print(len(select))
all_indices_set = set(range(1000))

TP = len(truth.intersection(select))
FP = len(select.difference(truth))
FN = len(truth.difference(select))
TN = len(all_indices_set.difference(truth).intersection(all_indices_set.difference(select)))


sensitivity = TP / (TP + FN)
specificity = TN / (FP + TN)

print("Sensitivity: ", sensitivity, "Specificity: ", specificity)